### Update projected starting lineups

In [3]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 14 teams with confirmed lineups


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [5]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

# usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_9318/2207245272.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Jalen Brunson,Over,28.5,-137,2025-11-22,2025-11-22T20:25:33Z
1,Underdog,player_points,Jalen Brunson,Under,28.5,-137,2025-11-22,2025-11-22T20:25:33Z
2,Underdog,player_points,Karl-Anthony Towns,Over,22.5,-137,2025-11-22,2025-11-22T20:25:33Z
3,Underdog,player_points,Karl-Anthony Towns,Under,22.5,-137,2025-11-22,2025-11-22T20:25:33Z
4,Underdog,player_points,Franz Wagner,Over,23.5,-137,2025-11-22,2025-11-22T20:25:33Z


### Top EVs for single bets

In [8]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 139 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
189,Jerami Grant,DraftKings,22.5,17.10,Under,-111,1,6.09,0.676,Med
1060,Bennedict Mathurin,BetMGM,21.5,26.02,Over,110,1,5.92,0.538,High
854,Alperen Sengun,BetRivers,24.5,28.14,Over,112,0,5.09,0.455,High
444,Keyonte George,FanDuel,18.5,23.11,Over,100,1,5.01,0.501,High
1140,Tre Jones,BetMGM,9.5,13.85,Over,-110,1,4.76,0.524,Med


## Top EVs for 2 leg bets

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 71 players...
Processing 63 players...
Generated 1812 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
400,Miles McBride,Coby White,8.5,20.5,13.06,25.74,0.764,0.825,over,over,1,85.31,0.427,High,Med
490,Jonathan Isaac,Peyton Watson,3.5,12.5,6.04,9.44,0.711,0.753,over,under,0,57.33,0.287,Low,Low
1488,Myles Turner,Cameron Johnson,15.5,13.5,12.32,10.05,0.697,0.728,under,under,0,49.16,0.246,High,Med
622,Landry Shamet,Duncan Robinson,9.5,10.5,12.17,13.07,0.697,0.664,over,over,0,36.02,0.180,Med,High
96,Karl-Anthony Towns,Santi Aldama,22.5,17.5,25.82,14.94,0.680,0.660,over,under,0,32.07,0.160,High,High
358,Mitchell Robinson,Bobby Portis,4.5,15.5,6.43,13.15,0.660,0.651,over,under,0,26.26,0.131,Low,High
236,Jordan Clarkson,Jeremiah Fears,10.5,16.5,12.76,18.78,0.645,0.640,over,over,0,21.41,0.107,High,High
943,Nickeil Alexander-Walker,Zach LaVine,18.5,18.5,20.56,20.85,0.619,0.639,over,over,0,16.24,0.081,High,High
48,Jalen Brunson,DeMar DeRozan,28.5,17.5,26.59,15.35,0.618,0.628,under,under,0,14.12,0.071,High,High
293,Mikal Bridges,Alex Sarr,15.5,17.5,17.39,19.36,0.612,0.610,over,over,0,9.89,0.049,High,High


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 82 players...
Error getting prediction for Keegan Murray: float division by zero
Processing 71 players...
Generated 2306 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
518,Miles McBride,Coby White,8.5,20.5,13.06,25.74,over,over,0.764,0.825,0.6177,0.186,0.247,0.296,85.31,0.427,1,6.34,5.61,High,Med,"(0.6, 25.5)","(14.7, 36.7)",0.05,0,85.3
1822,Jalen Duren,Peyton Watson,17.5,12.5,22.25,9.44,over,under,0.755,0.753,0.5569,0.177,0.175,0.234,67.07,0.335,0,6.87,4.49,High,Low,"(8.8, 35.7)","(0.6, 18.2)",0.05,0,67.1
473,Landry Shamet,Tobias Harris,9.0,10.5,12.17,14.89,over,over,0.730,0.751,0.5369,0.152,0.173,0.214,61.07,0.305,0,5.17,6.50,Med,High,"(2.0, 22.3)","(2.2, 27.6)",0.05,0,61.1
100,Karl-Anthony Towns,Ausar Thompson,21.5,9.5,25.82,13.38,over,over,0.729,0.738,0.5269,0.151,0.160,0.204,58.08,0.290,0,7.10,6.09,High,High,"(11.9, 39.7)","(1.4, 25.3)",0.05,0,58.1
734,Jonathan Isaac,Cameron Johnson,3.5,13.5,6.04,10.05,over,under,0.711,0.728,0.5071,0.133,0.150,0.183,52.13,0.261,0,4.57,5.70,Low,Med,"(0.0, 15.0)","(0.0, 21.2)",0.05,0,52.1
761,Jalen Johnson,Myles Turner,22.5,15.5,25.33,12.32,over,under,0.666,0.697,0.4549,0.088,0.119,0.130,36.47,0.182,0,6.61,6.15,High,High,"(12.4, 38.3)","(0.3, 24.4)",0.05,0,36.5
1921,Duncan Robinson,Santi Aldama,10.5,17.5,13.07,14.94,over,under,0.664,0.660,0.4296,0.086,0.082,0.104,28.89,0.144,0,6.07,6.19,High,High,"(1.2, 25.0)","(2.8, 27.1)",0.05,0,28.9
672,Mitchell Robinson,Jamal Murray,4.5,22.5,6.43,25.47,over,over,0.660,0.659,0.4258,0.082,0.081,0.100,27.75,0.139,0,4.70,7.26,Low,High,"(0.0, 15.6)","(11.2, 39.7)",0.05,0,27.8
387,Jordan Clarkson,Jeremiah Fears,10.5,16.5,12.76,18.78,over,over,0.645,0.640,0.4047,0.067,0.062,0.079,21.41,0.107,0,6.04,6.38,High,High,"(0.9, 24.6)","(6.3, 31.3)",0.05,0,21.4
307,Tristan da Silva,Zach LaVine,12.5,18.5,14.88,20.85,over,over,0.642,0.639,0.4022,0.064,0.061,0.076,20.66,0.103,0,6.52,6.59,High,High,"(2.1, 27.7)","(7.9, 33.8)",0.05,0,20.7


## 3 leg parlay

### Underdog picks

In [12]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 71 players...
Processing 63 players...
Generated 38801 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
12517,Miles McBride,Coby White,Peyton Watson,8.5,20.5,12.5,13.06,25.74,9.44,0.764,0.825,0.753,over,over,under,0,156.16,0.312,High,Med,Low
14304,Jonathan Isaac,Myles Turner,Cameron Johnson,3.5,15.5,13.5,6.04,12.32,10.05,0.711,0.697,0.728,over,under,under,0,94.81,0.190,Low,High,Med
2321,Karl-Anthony Towns,Landry Shamet,Duncan Robinson,22.5,9.5,10.5,25.82,12.17,13.07,0.680,0.697,0.664,over,over,over,0,69.94,0.140,High,Med,High
11414,Mitchell Robinson,Bobby Portis,Santi Aldama,4.5,15.5,17.5,6.43,13.15,14.94,0.660,0.651,0.660,over,under,under,0,53.15,0.106,Low,High,High
7840,Jordan Clarkson,Jeremiah Fears,Zach LaVine,10.5,16.5,18.5,12.76,18.78,20.85,0.645,0.640,0.639,over,over,over,0,42.54,0.085,High,High,High


### Prizepicks picks

In [15]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 82 players...
Error getting prediction for Keegan Murray: float division by zero
Processing 71 players...
Generated 55858 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
18091,Miles McBride,Coby White,Jalen Duren,8.5,20.5,17.5,13.06,25.74,22.25,0.764,0.825,0.755,over,over,over,1,157.00,0.314,High,Med,High
52419,Tobias Harris,Ausar Thompson,Peyton Watson,10.5,9.5,12.5,14.89,13.38,9.44,0.751,0.738,0.753,over,over,under,0,125.07,0.250,High,High,Low
2705,Karl-Anthony Towns,Landry Shamet,Cameron Johnson,21.5,9.0,13.5,25.82,12.17,10.05,0.729,0.730,0.728,over,over,under,0,109.02,0.218,High,Med,Med
22739,Jonathan Isaac,Jalen Johnson,Myles Turner,3.5,22.5,15.5,6.04,25.33,12.32,0.711,0.666,0.697,over,over,under,0,78.24,0.156,Low,High,High
22255,Mitchell Robinson,Duncan Robinson,Santi Aldama,4.5,10.5,17.5,6.43,13.07,14.94,0.660,0.664,0.660,over,over,under,0,56.17,0.112,Low,High,High


In [14]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)